# Atividade Ponderada - Bag of Words (BoW)
**Módulo 6 - Processamento de Linguagem Natural**

Inteli - Instituto de Tecnologia e Liderança  
Curso: Sistemas de Informação  
Eixo: Computação
Professor: Bryan Kano

Aluno: Matheus Henrique Scapolan Silva


## 1. Contexto e Motivação da Atividade

O Módulo 6 do curso de Sistemas de Informação do Inteli é dedicado ao desenvolvimento de aplicações que processam e analisam textos por meio de inteligência artificial.

Dentro desse contexto, esta atividade ponderada tem como objetivo consolidar a compreensão do modelo **Bag of Words (BoW)**, que é a representação vetorial de textos mais utilizada como ponto de partida histórico em PLN. Conforme apresentado por Jurafsky e Martin em *Speech and Language Processing* (2024, Stanford University), antes de qualquer algoritmo de aprendizado de máquina operar sobre linguagem, o texto precisa ser convertido em uma representação numérica de dimensão fixa. O BoW cumpre exatamente essa função ao mapear documentos para vetores de contagem de palavras, descartando a ordem mas preservando a frequência de ocorrência.

A atividade propõe a leitura do tutorial disponível no Kaggle (*Bag-of-Words Model for Beginners*, de Vipul Gandhi) e o desenvolvimento de um código Python organizado no Google Colab, com testes sobre 25 frases em inglês e 25 frases em português. Para o processamento em português, são utilizadas as bibliotecas **NLTK** e **spaCy**, conforme abordado nas aulas do Prof. Bryan Kano sobre pré-processamento textual.


## 2. Instalação e Configuração do Ambiente

As bibliotecas utilizadas nesta atividade cobrem dois universos complementares. Para o inglês, **scikit-learn** fornece os vetorizadores `CountVectorizer`, `TfidfVectorizer` e `HashingVectorizer`, além do `Tokenizer` do **Keras/TensorFlow**. Para o português, **NLTK** disponibiliza lista de stopwords, o stemmer RSLP (desenvolvido especificamente para o português brasileiro) e utilitários de tokenização, enquanto **spaCy** com o modelo `pt_core_news_sm` realiza lematização neural contextualizada que é uma técnica mais precisa do que o stemming porque produz formas canônicas válidas no léxico (Bird, Klein e Loper, *Natural Language Processing with Python*, O'Reilly, 2009).

A célula abaixo instala e configura tudo o que será necessário ao longo do notebook.


In [1]:
# Instalação das dependências
!pip install -q scikit-learn tensorflow nltk spacy

# Download dos recursos do NLTK
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('rslp', quiet=True)

# Download do modelo neural de português do spaCy
import subprocess
subprocess.run(["python", "-m", "spacy", "download", "pt_core_news_sm"], capture_output=True)

print("Ambiente configurado com sucesso.")



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Ambiente configurado com sucesso.


## 3. Importações

Todas as bibliotecas são importadas de uma vez para manter o notebook organizado e facilitar a rastreabilidade das dependências.


In [2]:
# Vetorizadores do scikit-learn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer

# Keras — tokenização e hashing trick
from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence, hashing_trick

# NLTK — pré-processamento em português
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer

# spaCy — lematização neural em português
import spacy

# Utilitários
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

nlp_pt = spacy.load("pt_core_news_sm")
stop_pt = set(stopwords.words("portuguese"))
stemmer = RSLPStemmer()

print("Todas as importações realizadas com sucesso.")


Todas as importações realizadas com sucesso.


## 4. O que é o Bag of Words e por que ele importa

O modelo Bag of Words representa cada documento como um vetor de dimensão igual ao tamanho do vocabulário do corpus, onde cada posição registra a frequência de ocorrência da palavra correspondente. A denominação "saco de palavras" é intencional, uma vez que a ordem das palavras é completamente descartada, e apenas a frequência importa.

Formalmente, dado um vocabulário $V = \{w_1, w_2, \ldots, w_n\}$ construído sobre um corpus $\mathcal{D}$, o vetor BoW de um documento $d$ é:

$$x^{(d)} = \left(x_1^{(d)},\, x_2^{(d)},\, \ldots,\, x_n^{(d)}\right) \in \mathbb{Z}_{\geq 0}^n, \quad x_k^{(d)} = \text{tf}(w_k, d)$$

O principal efeito colateral desta representação é a **esparsidade** (propriedade, onde um conjunto de dados, matriz ou vetor contém uma grande quantidade de valores zero ou ausentes.), visto que se o vocabulário tem 50 000 tokens e um documento usa apenas 200 palavras distintas, a esparsidade do vetor é $1 - 200/50000 = 99{,}6\%$. Na prática, as implementações do scikit-learn utilizam matrizes esparsas no formato CSR (*Compressed Sparse Row*) para economizar memória.

Apesar de suas limitações conhecidas, como perda de ordem, ausência de semântica entre sinônimos, alta dimensionalidade e incapacidade de lidar com polissemia, o BoW permanece como uma base conceitual indispensável e ainda é utilizado em produção em sistemas de classificação de texto de grande escala.


## 5. Corpus de Testes - 25 Frases em Inglês

As 25 frases abaixo foram elaboradas para cobrir diferentes domínios semânticos (tecnologia, gastronomia, esporte, ciência, cotidiano) e também para incluir casos de interesse analítico como, frases com negação, repetição de palavras, vocabulário raro e sobreposição intencional de termos entre documentos. Esse mix permite observar de forma rica o comportamento dos diferentes esquemas de vetorização.


In [3]:
sentences_en = [
    # tecnologia e IA
    "Neural networks learn patterns from large amounts of labeled training data",
    "The transformer architecture revolutionized natural language processing tasks",
    "Gradient descent optimizes the model weights by minimizing the loss function",
    "Overfitting occurs when a model memorizes training data instead of generalizing",
    "Transfer learning allows pretrained models to be fine-tuned on smaller datasets",
    # gastronomia
    "The chef prepared a slow-cooked beef stew with seasonal vegetables and red wine",
    "Sourdough bread requires a fermented starter culture to develop its tangy flavor",
    "Umami is often described as the fifth basic taste alongside sweet sour salty and bitter",
    "Cold brew coffee is steeped in cold water for twelve to twenty-four hours",
    "Tempering chocolate requires precise temperature control to achieve a glossy snap",
    # esporte e movimento
    "Marathon runners often hit a wall around the twentieth mile due to glycogen depletion",
    "The goalkeeper made three consecutive saves in the final minutes of extra time",
    "Rock climbing demands both physical strength and precise footwork on the wall",
    "Swimming is considered a low-impact exercise that benefits cardiovascular health",
    "The athlete broke the world record by finishing the sprint in under nine seconds",
    # ciência e natureza
    "Photosynthesis converts sunlight carbon dioxide and water into glucose and oxygen",
    "Black holes warp spacetime so severely that not even light can escape their gravity",
    "The human microbiome contains trillions of bacteria that influence digestion and immunity",
    "Plate tectonics explains why earthquakes and volcanoes cluster along specific boundaries",
    "Quantum entanglement allows two particles to share a correlated state across distances",
    # cotidiano com negação e variação
    "The meeting was not as productive as the team had initially hoped it would be",
    "She did not enjoy the noisy restaurant despite the excellent quality of the food",
    "Early morning routines help people maintain focus and energy throughout the day",
    "Reading fiction novels improves empathy by exposing readers to diverse perspectives",
    "The city council voted against building a new highway through the historic district",
]

print(f"Total de frases em inglês: {len(sentences_en)}")
for i, s in enumerate(sentences_en, 1):
    print(f"  [{i:02d}] {s}")

Total de frases em inglês: 25
  [01] Neural networks learn patterns from large amounts of labeled training data
  [02] The transformer architecture revolutionized natural language processing tasks
  [03] Gradient descent optimizes the model weights by minimizing the loss function
  [04] Overfitting occurs when a model memorizes training data instead of generalizing
  [05] Transfer learning allows pretrained models to be fine-tuned on smaller datasets
  [06] The chef prepared a slow-cooked beef stew with seasonal vegetables and red wine
  [07] Sourdough bread requires a fermented starter culture to develop its tangy flavor
  [08] Umami is often described as the fifth basic taste alongside sweet sour salty and bitter
  [09] Cold brew coffee is steeped in cold water for twelve to twenty-four hours
  [10] Tempering chocolate requires precise temperature control to achieve a glossy snap
  [11] Marathon runners often hit a wall around the twentieth mile due to glycogen depletion
  [12] The g

## 6. Vetorização com CountVectorizer (scikit-learn)

O `CountVectorizer` realiza três operações em sequência: tokenização do texto (separação em palavras), construção do vocabulário a partir do corpus de treino e codificação de cada documento como um vetor de contagens inteiras. A chamada `fit()` aprende o vocabulário; `transform()` aplica a codificação.

A matriz resultante tem dimensão $m \times n$, onde $m$ é o número de documentos e $n$ é o tamanho do vocabulário. Essa estrutura é denominada **Matriz Documento-Termo** (MDT) e é a entrada direta para algoritmos como Naïve Bayes, SVM e KNN (Manning et al., 2008).


In [4]:
# Criando e ajustando o vetorizador
cv = CountVectorizer()
cv.fit(sentences_en)

vocab_sorted = sorted(cv.vocabulary_)
print(f"Tamanho do vocabulário: {len(vocab_sorted)} tokens únicos")
print(f"Primeiros 20 tokens: {vocab_sorted[:20]}")

# Transformando os documentos
X_count = cv.transform(sentences_en)
print(f"\nFormato da Matriz Documento-Termo: {X_count.shape}")
print(f"Tipo da matriz: {type(X_count)}")
print(f"Valores não-zero: {X_count.nnz} de {X_count.shape[0] * X_count.shape[1]} possíveis")
sparsidade = 1 - X_count.nnz / (X_count.shape[0] * X_count.shape[1])
print(f"Esparsidade: {sparsidade:.2%}")

Tamanho do vocabulário: 239 tokens únicos
Primeiros 20 tokens: ['achieve', 'across', 'against', 'allows', 'along', 'alongside', 'amounts', 'and', 'architecture', 'around', 'as', 'athlete', 'bacteria', 'basic', 'be', 'beef', 'benefits', 'bitter', 'black', 'both']

Formato da Matriz Documento-Termo: (25, 239)
Tipo da matriz: <class 'scipy.sparse._csr.csr_matrix'>
Valores não-zero: 289 de 5975 possíveis
Esparsidade: 95.16%


In [5]:
# Exibindo a MDT como DataFrame para os primeiros 5 documentos e 15 palavras
feature_names = cv.get_feature_names_out()
df_count = pd.DataFrame(X_count.toarray(), columns=feature_names)

# Selecionar colunas com pelo menos 1 ocorrência nos primeiros 5 docs
subset = df_count.iloc[:5]
cols_com_valor = subset.loc[:, (subset > 0).any()].columns[:15]

print("Matriz Documento-Termo (primeiros 5 documentos, até 15 colunas com valores):")
print(df_count.iloc[:5][cols_com_valor].to_string())

Matriz Documento-Termo (primeiros 5 documentos, até 15 colunas com valores):
   allows  amounts  architecture  be  by  data  datasets  descent  fine  from  function  generalizing  gradient  instead  labeled
0       0        1             0   0   0     1         0        0     0     1         0             0         0        0        1
1       0        0             1   0   0     0         0        0     0     0         0             0         0        0        0
2       0        0             0   0   1     0         0        1     0     0         1             0         1        0        0
3       0        0             0   0   0     1         0        0     0     0         0             1         0        1        0
4       1        0             0   1   0     0         1        0     1     0         0             0         0        0        0


In [10]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Corpus de treino
corpus = [
    "The coffee contains fermented grains",
    "Fermented food improves digestion",
    "The model learns from data",
    "Natural language processing transforms text"
]

# Treinando o CountVectorizer
cv = CountVectorizer()
cv.fit(corpus)

# Frase com palavras OOV
frase_oov = [
    "The zythologist analyzed rare fermented beverages from ancient civilizations"
]

# Vetorizando a frase
vetor_oov = cv.transform(frase_oov)

# Vocabulário aprendido
feature_names = cv.get_feature_names_out()

# Tokens da frase
tokens = frase_oov[0].lower().split()

# Palavras reconhecidas e OOV
palavras_reconhecidas = [
    palavra for palavra in tokens
    if palavra in cv.vocabulary_
]

palavras_oov = [
    palavra for palavra in tokens
    if palavra not in cv.vocabulary_
]

# Vetor apenas com valores não-zero
vetor = vetor_oov.toarray()[0]

valores_nao_zero = {
    palavra: int(valor)
    for palavra, valor in zip(feature_names, vetor)
    if valor > 0
}

# Exibição
print("Frase analisada:")
print(frase_oov[0])

print("\nPalavras reconhecidas:")
print(palavras_reconhecidas)

print("\nPalavras OOV (ignoradas):")
print(palavras_oov)

print("\nVetor (somente valores não-zero):")
print(valores_nao_zero)

# DataFrame opcional
df = pd.DataFrame(vetor.reshape(1, -1), columns=feature_names)

print("\nRepresentação vetorial completa:")
print(df)

Frase analisada:
The zythologist analyzed rare fermented beverages from ancient civilizations

Palavras reconhecidas:
['the', 'fermented', 'from']

Palavras OOV (ignoradas):
['zythologist', 'analyzed', 'rare', 'beverages', 'ancient', 'civilizations']

Vetor (somente valores não-zero):
{'fermented': 1, 'from': 1, 'the': 1}

Representação vetorial completa:
   coffee  contains  data  digestion  fermented  food  from  grains  improves  \
0       0         0     0          0          1     0     1       0         0   

   language  learns  model  natural  processing  text  the  transforms  
0         0       0      0        0           0     0    1           0  


## 7. Vetorização com TfidfVectorizer (scikit-learn)

O modelo Bag of Words tradicional utiliza apenas a contagem bruta de palavras. O problema dessa abordagem é que termos muito frequentes no corpus, como “the”, “is” e “and”, acabam recebendo grande importância mesmo carregando pouca informação semântica.

Para reduzir esse efeito, utiliza-se o modelo **TF-IDF** (*Term Frequency — Inverse Document Frequency*), que aumenta o peso de palavras importantes em um documento e reduz o peso de palavras muito comuns no corpus inteiro.

De forma simplificada

- TF mede quantas vezes uma palavra aparece em um documento
- IDF mede o quão rara essa palavra é no conjunto total de documentos

Assim, palavras raras e específicas recebem pesos maiores, enquanto palavras extremamente frequentes recebem pesos menores.

Formalmente

$$
\mathrm{tfidf}(w,d,\mathcal{D}) =
\mathrm{tf}(w,d)
\times
\log\frac{|\mathcal{D}|}{|\{d' \in \mathcal{D}: w \in d'\}|}
$$

Na prática, isso torna os vetores mais informativos para tarefas de classificação, agrupamento e cálculo de similaridade textual. Além disso, o `TfidfVectorizer` do scikit-learn aplica normalização vetorial automática, permitindo comparar documentos utilizando similaridade por cosseno.

In [11]:
tfidf = TfidfVectorizer()
tfidf.fit(sentences_en)
X_tfidf = tfidf.transform(sentences_en)

feature_names_tfidf = tfidf.get_feature_names_out()
idf_values = tfidf.idf_

# Ordenar por IDF - palavras com IDF mais alto são as mais raras no corpus
idf_series = pd.Series(idf_values, index=feature_names_tfidf).sort_values(ascending=False)

print("=== 10 palavras mais discriminativas (IDF mais alto) ===")
print(idf_series.head(10).to_string())
print()
print("=== 10 palavras menos discriminativas (IDF mais baixo) ===")
print(idf_series.tail(10).to_string())


=== 10 palavras mais discriminativas (IDF mais alto) ===
achieve         3.564949
across          3.564949
against         3.564949
along           3.564949
alongside       3.564949
amounts         3.564949
architecture    3.564949
basic           3.564949
around          3.564949
bacteria        3.564949

=== 10 palavras menos discriminativas (IDF mais baixo) ===
on      3.159484
by      2.871802
that    2.871802
in      2.871802
is      2.871802
not     2.871802
of      2.466337
and     2.178655
to      2.178655
the     1.619039


In [12]:
# Comparando TF-IDF da primeira frase
frase_idx = 0
vetor_frase = X_tfidf[frase_idx].toarray()[0]
termos_com_peso = sorted(zip(feature_names_tfidf, vetor_frase), key=lambda x: x[1], reverse=True)
termos_nao_zero = [(t, round(v, 4)) for t, v in termos_com_peso if v > 0]

print(f"Frase analisada: '{sentences_en[frase_idx]}'")
print()
print("Tokens com seus pesos TF-IDF (ordenados do mais ao menos relevante):")
for token, peso in termos_nao_zero:
    print(f"  {token:<20} {peso}")

Frase analisada: 'Neural networks learn patterns from large amounts of labeled training data'

Tokens com seus pesos TF-IDF (ordenados do mais ao menos relevante):
  amounts              0.3154
  from                 0.3154
  labeled              0.3154
  large                0.3154
  learn                0.3154
  networks             0.3154
  neural               0.3154
  patterns             0.3154
  data                 0.2796
  training             0.2796
  of                   0.2182


## 8. Similaridade por Cosseno entre Documentos

Após transformar os textos em vetores numéricos, torna-se possível comparar documentos matematicamente. Uma das métricas mais utilizadas em PLN é a similaridade por cosseno, que mede o grau de semelhança entre dois vetores com base no ângulo formado entre eles.

Diferente da distância euclidiana, a similaridade por cosseno não é afetada diretamente pelo tamanho do documento, tornando a comparação mais adequada para textos de comprimentos diferentes.

De forma intuitiva

- valores próximos de 1 indicam documentos muito parecidos
- valores próximos de 0 indicam pouca relação entre os textos

Formalmente

$$
\cos(u,v)=\frac{u\cdot v}{\|u\|\cdot\|v\|}
$$

Na prática, essa métrica é amplamente utilizada em mecanismos de busca, sistemas de recomendação, agrupamento de documentos e análise de similaridade textual.

In [13]:
# Calculando a matriz de similaridade usando os vetores TF-IDF
sim_matrix = cosine_similarity(X_tfidf)

# Encontrar o par mais similar (excluindo a diagonal)
np.fill_diagonal(sim_matrix, 0)
idx_max = np.unravel_index(np.argmax(sim_matrix), sim_matrix.shape)
i, j = idx_max

print(f"Par de frases mais similar em inglês (cos = {sim_matrix[i, j]:.4f}):")
print(f"  Frase {i+1}: {sentences_en[i]}")
print(f"  Frase {j+1}: {sentences_en[j]}")
print()

# Frases mais dissimilares
np.fill_diagonal(sim_matrix, 1)
idx_min = np.unravel_index(np.argmin(sim_matrix), sim_matrix.shape)
i2, j2 = idx_min
print(f"Par de frases mais dissimilar em inglês (cos = {sim_matrix[i2, j2]:.4f}):")
print(f"  Frase {i2+1}: {sentences_en[i2]}")
print(f"  Frase {j2+1}: {sentences_en[j2]}")

Par de frases mais similar em inglês (cos = 0.2175):
  Frase 1: Neural networks learn patterns from large amounts of labeled training data
  Frase 4: Overfitting occurs when a model memorizes training data instead of generalizing

Par de frases mais dissimilar em inglês (cos = 0.0000):
  Frase 1: Neural networks learn patterns from large amounts of labeled training data
  Frase 2: The transformer architecture revolutionized natural language processing tasks


## 9. Vetorização com HashingVectorizer (scikit-learn)

O `HashingVectorizer` elimina a necessidade de manter o vocabulário em memória ao mapear cada token diretamente para uma posição de vetor via função hash. Isso resolve o problema de escalabilidade com vocabulários muito grandes (acima de $10^6$ tokens), ao custo de duas limitações: colisões de hash (palavras distintas mapeadas para a mesma posição) e a impossibilidade de reverter o vetor para os tokens originais.

O parâmetro `n_features` define o espaço de hash; valores pequenos aumentam as colisões.


In [16]:
# Vetorização com HashingVectorizer
hv = HashingVectorizer(
    n_features=2**10,
    alternate_sign=False
)

X_hash = hv.transform(sentences_en)

# Frases para comparação
frase_a = sentences_en[0]
frase_b = sentences_en[4]

# Vetores
vetor_a = X_hash[0]
vetor_b = X_hash[4]

# Similaridade de cosseno
cos_hash = cosine_similarity(vetor_a, vetor_b)[0][0]

# Quantidade de posições não-zero
nao_zero_a = np.count_nonzero(vetor_a.toarray())
nao_zero_b = np.count_nonzero(vetor_b.toarray())

# Resultados
print("Formato da matriz hash:")
print(X_hash.shape)

print("\nNúmero de features:")
print(2**10)

print("\nFrase 1:")
print(frase_a)

print("\nFrase 2:")
print(frase_b)

print("\nSimilaridade de cosseno:")
print(f"{cos_hash:.4f}")

print("\nPosições não-zero:")
print(f"Frase 1: {nao_zero_a}")
print(f"Frase 2: {nao_zero_b}")

# Visualização parcial dos vetores
df_hash = pd.DataFrame({
    "Hash Index": np.arange(20),
    "Frase 1": vetor_a.toarray()[0][:20],
    "Frase 2": vetor_b.toarray()[0][:20]
})

print("\nPrimeiras posições do vetor hash:")
print(df_hash)

print("\nObservação:")
print("O HashingVectorizer utiliza função hash e não mantém vocabulário explícito.")

Formato da matriz hash:
(25, 1024)

Número de features:
1024

Frase 1:
Neural networks learn patterns from large amounts of labeled training data

Frase 2:
Transfer learning allows pretrained models to be fine-tuned on smaller datasets

Similaridade de cosseno:
0.0000

Posições não-zero:
Frase 1: 11
Frase 2: 12

Primeiras posições do vetor hash:
    Hash Index  Frase 1  Frase 2
0            0      0.0      0.0
1            1      0.0      0.0
2            2      0.0      0.0
3            3      0.0      0.0
4            4      0.0      0.0
5            5      0.0      0.0
6            6      0.0      0.0
7            7      0.0      0.0
8            8      0.0      0.0
9            9      0.0      0.0
10          10      0.0      0.0
11          11      0.0      0.0
12          12      0.0      0.0
13          13      0.0      0.0
14          14      0.0      0.0
15          15      0.0      0.0
16          16      0.0      0.0
17          17      0.0      0.0
18          18      0.0  

## 10. Bag of Words com Keras - Tokenizer e Hashing Trick

Além das ferramentas do scikit-learn, o ecossistema do Keras também oferece utilitários para pré-processamento textual voltados para aplicações de Deep Learning.

O `Tokenizer` cria automaticamente um índice numérico para as palavras do corpus com base na frequência de ocorrência. Depois disso, os textos podem ser convertidos em representações numéricas adequadas para redes neurais.

O Keras permite diferentes formas de codificação textual, como

- representação binária
- contagem de palavras
- TF-IDF
- frequência relativa

Outra funcionalidade importante é o `hashing_trick`, que aplica uma estratégia semelhante ao `HashingVectorizer`. Nesse caso, as palavras são transformadas em índices numéricos utilizando funções hash, sem necessidade de armazenar um vocabulário explícito em memória.

Essas ferramentas são amplamente utilizadas em pipelines de Deep Learning para PLN, principalmente como etapa de preparação de dados para modelos baseados em embeddings, LSTM e arquiteturas Transformer.

In [19]:
# Tokenizer do Keras
tokenizer_keras = Tokenizer()
tokenizer_keras.fit_on_texts(sentences_en)

print(f"Vocabulário aprendido pelo Keras Tokenizer: {len(tokenizer_keras.word_index)} tokens")
print()
print("10 palavras mais frequentes (índice menor = mais frequente):")
top10 = sorted(tokenizer_keras.word_counts.items(), key=lambda x: x[1], reverse=True)[:10]
for palavra, contagem in top10:
    print(f"  {palavra:<20} contagem={contagem}  índice={tokenizer_keras.word_index[palavra]}")

Vocabulário aprendido pelo Keras Tokenizer: 240 tokens

10 palavras mais frequentes (índice menor = mais frequente):
  the                  contagem=21  índice=1
  a                    contagem=8  índice=2
  and                  contagem=8  índice=3
  to                   contagem=7  índice=4
  of                   contagem=5  índice=5
  by                   contagem=3  índice=6
  is                   contagem=3  índice=7
  as                   contagem=3  índice=8
  in                   contagem=3  índice=9
  that                 contagem=3  índice=10


In [20]:
# Codificação dos textos com Keras Tokenizer

modos = ["binary", "count", "tfidf", "freq"]

for modo in modos:

    matriz = tokenizer_keras.texts_to_matrix(
        sentences_en[:3],
        mode=modo
    )

    df_modo = pd.DataFrame(
        matriz[:, :10].round(4),
        columns=[f"token_{i}" for i in range(10)]
    )

    print(f"\nModo: {modo}")
    print("Shape da matriz:", matriz.shape)

    print("\nRepresentação dos 3 primeiros documentos:")
    print(df_modo)


# Hashing Trick do Keras

frase = sentences_en[0]

tokens = text_to_word_sequence(frase)

tokens_unicos = sorted(set(tokens))

hash_space = round(len(tokens_unicos) * 1.3)

hash_resultado = hashing_trick(
    frase,
    n=hash_space,
    hash_function="md5"
)

df_hash = pd.DataFrame({
    "Token": tokens,
    "Hash Index": hash_resultado
})

print("\nFrase analisada:")
print(frase)

print("\nTokens únicos:")
print(tokens_unicos)

print("\nEspaço hash:")
print(hash_space)

print("\nMapeamento token -> índice hash:")
print(df_hash)


Modo: binary
Shape da matriz: (3, 241)

Representação dos 3 primeiros documentos:
   token_0  token_1  token_2  token_3  token_4  token_5  token_6  token_7  \
0      0.0      0.0      0.0      0.0      0.0      1.0      0.0      0.0   
1      0.0      1.0      0.0      0.0      0.0      0.0      0.0      0.0   
2      0.0      1.0      0.0      0.0      0.0      0.0      1.0      0.0   

   token_8  token_9  
0      0.0      0.0  
1      0.0      0.0  
2      0.0      0.0  

Modo: count
Shape da matriz: (3, 241)

Representação dos 3 primeiros documentos:
   token_0  token_1  token_2  token_3  token_4  token_5  token_6  token_7  \
0      0.0      0.0      0.0      0.0      0.0      1.0      0.0      0.0   
1      0.0      1.0      0.0      0.0      0.0      0.0      0.0      0.0   
2      0.0      2.0      0.0      0.0      0.0      0.0      1.0      0.0   

   token_8  token_9  
0      0.0      0.0  
1      0.0      0.0  
2      0.0      0.0  

Modo: tfidf
Shape da matriz: (3, 241)

R

## 11. N-gramas — Capturando Contexto Local

Uma das principais limitações do modelo Bag-of-Words é que ele ignora completamente a ordem das palavras. Isso significa que frases semanticamente diferentes podem acabar tendo representações muito parecidas apenas porque compartilham os mesmos tokens.

Os modelos de **n-gramas** reduzem parcialmente esse problema ao considerar sequências consecutivas de palavras como unidades do vocabulário.

Um:

- **unigrama** representa 1 palavra
- **bigrama** representa 2 palavras consecutivas
- **trigrama** representa 3 palavras consecutivas

Por exemplo:

| Texto | Representação |
|---|---|
| "not good" | ["not", "good", "not good"] |

Nesse caso, o bigrama `"not good"` captura um contexto negativo que seria perdido ao analisar apenas as palavras individualmente.

Essa abordagem é especialmente importante em tarefas de:

- análise de sentimentos
- classificação de texto
- detecção de contexto local
- sistemas de recomendação textual

O scikit-learn permite trabalhar com n-gramas diretamente pelo parâmetro:

```python
ngram_range=(min_n, max_n)
```

Por exemplo:

```python
ngram_range=(1, 2)
```

significa:

- incluir unigramas
- incluir bigramas

simultaneamente no vocabulário.

Assim, o modelo consegue preservar parte da estrutura textual e diferenciar frases como:

- `"not good"`
- `"very good"`

mesmo que ambas contenham a palavra `"good"`.

In [ ]:
# BoW com unigramas e bigramas combinados
cv_bigram = CountVectorizer(ngram_range=(1, 2), max_features=200)
cv_bigram.fit(sentences_en)
X_bigram = cv_bigram.transform(sentences_en)

vocab_bigram = sorted(cv_bigram.vocabulary_.keys())
bigramas = [t for t in vocab_bigram if " " in t]

print(f"Vocabulário unigrama+bigrama: {len(vocab_bigram)} tokens")
print(f"Somente bigramas: {len(bigramas)}")
print()
print("Exemplos de bigramas capturados:")
for bg in bigramas[:15]:
    print(f"  '{bg}'")

Vocabulário unigrama+bigrama: 200 tokens
Somente bigramas: 89

Exemplos de bigramas capturados:
  'achieve glossy'
  'across distances'
  'against building'
  'allows pretrained'
  'allows two'
  'along specific'
  'alongside sweet'
  'amounts of'
  'as the'
  'bread requires'
  'brew coffee'
  'broke the'
  'building new'
  'by exposing'
  'by finishing'


In [23]:
# Efeito da negação — comparando BoW unigrama vs bigrama
negation_a = ["The movie was not good at all"]
negation_b = ["The movie was very good indeed"]

cv_uni = CountVectorizer(ngram_range=(1, 1))
cv_bi  = CountVectorizer(ngram_range=(1, 2))

for corpus_demo in [negation_a + negation_b]:
    cv_uni.fit(corpus_demo)
    cv_bi.fit(corpus_demo)

    v_uni_a = cv_uni.transform(negation_a).toarray()[0]
    v_uni_b = cv_uni.transform(negation_b).toarray()[0]
    cos_uni  = np.dot(v_uni_a, v_uni_b) / (np.linalg.norm(v_uni_a) * np.linalg.norm(v_uni_b))

    v_bi_a = cv_bi.transform(negation_a).toarray()[0]
    v_bi_b = cv_bi.transform(negation_b).toarray()[0]
    cos_bi  = np.dot(v_bi_a, v_bi_b) / (np.linalg.norm(v_bi_a) * np.linalg.norm(v_bi_b))

print("Efeito da negação na similaridade:")
print(f"  Frase A: '{negation_a[0]}'")
print(f"  Frase B: '{negation_b[0]}'")
print()
print(f"  Cosine com UNIGRAMAS: {cos_uni:.4f}  (frases parecem muito similares)")
print(f"  Cosine com BIGRAMAS:  {cos_bi:.4f}  (bigramas diferenciam melhor o contexto de negação)")

Efeito da negação na similaridade:
  Frase A: 'The movie was not good at all'
  Frase B: 'The movie was very good indeed'

  Cosine com UNIGRAMAS: 0.6172  (frases parecem muito similares)
  Cosine com BIGRAMAS:  0.5017  (bigramas diferenciam melhor o contexto de negação)


## 12. Corpus de Testes — 25 Frases em Português

As frases em português foram compostas para explorar desafios específicos do idioma: acentuação, contrações, morfologia rica (plural, conjugação verbal em múltiplos tempos) e termos técnicos. A presença de negações e frases sobre alimentação também conecta este corpus ao domínio da Zamp, tornando os testes mais aderentes ao projeto que esta sendo desenvolvido no módulo.


In [26]:
sentences_pt = [
    # tecnologia e dados
    "Redes neurais profundas extraem representações hierárquicas de dados brutos",
    "O gradiente descendente atualiza os pesos do modelo na direção oposta ao gradiente da perda",
    "Modelos de linguagem de grande escala foram pré-treinados em bilhões de tokens de texto",
    "A análise de sentimentos classifica opiniões de clientes em positivo neutro ou negativo",
    "Vetores de palavras densos capturam relações semânticas que o Bag of Words não consegue",
    # alimentação e consumo
    "O hambúrguer artesanal com carne wagyu foi avaliado como excelente pela maioria dos clientes",
    "A entrega demorou mais de uma hora e o pedido chegou completamente frio e sem condimentos",
    "O cardápio digital facilitou a personalização dos ingredientes no aplicativo do restaurante",
    "Clientes fiéis utilizam o programa de pontos para resgatar combos gratuitos mensalmente",
    "A nova sobremesa de chocolate belga superou as expectativas dos clientes mais exigentes",
    # cotidiano e negação
    "Não foi possível realizar o pagamento por aproximação devido à falha no terminal de vendas",
    "A loja não estava funcionando no horário indicado no site e isso gerou reclamações",
    "O atendimento foi rápido mas a qualidade dos ingredientes deixou muito a desejar",
    "Apesar do preço elevado o produto não correspondeu às expectativas dos consumidores",
    "A promoção relâmpago não foi comunicada com antecedência suficiente para os clientes habituais",
    # ciência e natureza
    "A fotossíntese converte dióxido de carbono água e luz solar em glicose e oxigênio molecular",
    "Buracos negros supermassivos habitam o centro da maioria das galáxias espirais observadas",
    "O microbioma intestinal influencia a imunidade o humor e o metabolismo do organismo humano",
    "Terremotos e vulcões concentram-se nas bordas das placas tectônicas em movimento constante",
    "O emaranhamento quântico permite correlações instantâneas entre partículas separadas por distâncias",
    # esporte e saúde
    "Corredores de maratona precisam gerenciar o glicogênio muscular para evitar a fadiga abrupta",
    "A natação é considerada o exercício mais completo por trabalhar grupos musculares distintos",
    "Atletas de alto rendimento monitoram a variabilidade da frequência cardíaca para otimizar o treino",
    "A prática regular de yoga melhora a flexibilidade o equilíbrio e a saúde mental dos praticantes",
    "Equipes de futebol utilizam dados de rastreamento por GPS para analisar o desempenho em campo",
]

print(f"Total de frases em português: {len(sentences_pt)}")
for i, s in enumerate(sentences_pt, 1):
    print(f"  [{i:02d}] {s}")

Total de frases em português: 25
  [01] Redes neurais profundas extraem representações hierárquicas de dados brutos
  [02] O gradiente descendente atualiza os pesos do modelo na direção oposta ao gradiente da perda
  [03] Modelos de linguagem de grande escala foram pré-treinados em bilhões de tokens de texto
  [04] A análise de sentimentos classifica opiniões de clientes em positivo neutro ou negativo
  [05] Vetores de palavras densos capturam relações semânticas que o Bag of Words não consegue
  [06] O hambúrguer artesanal com carne wagyu foi avaliado como excelente pela maioria dos clientes
  [07] A entrega demorou mais de uma hora e o pedido chegou completamente frio e sem condimentos
  [08] O cardápio digital facilitou a personalização dos ingredientes no aplicativo do restaurante
  [09] Clientes fiéis utilizam o programa de pontos para resgatar combos gratuitos mensalmente
  [10] A nova sobremesa de chocolate belga superou as expectativas dos clientes mais exigentes
  [11] Não foi

## 13. Pré-processamento Textual em Português

O texto em português apresenta desafios adicionais em relação ao inglês: sistema de acentuação rico, contrações (do, da, no, na, ao, pelo), morfologia verbal mais complexa e o risco de ambiguidade na remoção de acentos ("para" ≠ "pará", conforme mostrado nas aulas do Prof. Bryan).

O pipeline abaixo aplica quatro etapas em sequência, conforme a arquitetura clássica de pré-processamento apresentada nos slides do módulo: tokenização, normalização para minúsculas, remoção de stopwords e, em paralelo, **stemming** (RSLP) e **lematização** (spaCy). A comparação entre as duas técnicas explicita as diferenças qualitativas entre reduzir ao radical (rápido, impreciso) e reduzir ao lema (mais lento, produz formas válidas no léxico).


In [28]:
def preprocessar_pt(texto, usar_lema=True):
    """
    Pipeline de pré-processamento para português.
    Retorna lista de tokens limpos — lemas ou stems dependendo da flag.
    """
    tokens_brutos = word_tokenize(texto, language="portuguese")
    tokens_lower  = [t.lower() for t in tokens_brutos if t.isalpha()]
    sem_stopwords = [t for t in tokens_lower if t not in stop_pt]

    if usar_lema:
        doc = nlp_pt(" ".join(sem_stopwords))
        return [token.lemma_ for token in doc]
    else:
        return [stemmer.stem(t) for t in sem_stopwords]

# Demonstração em três frases representativas
demos = [sentences_pt[0], sentences_pt[6], sentences_pt[10]]

print("=== Comparação Stemming vs. Lematização ===\n")
for frase in demos:
    tokens_brutos = [t.lower() for t in word_tokenize(frase, language="portuguese") if t.isalpha()]
    sem_stop = [t for t in tokens_brutos if t not in stop_pt]
    stems  = [stemmer.stem(t) for t in sem_stop]
    doc_lema = nlp_pt(" ".join(sem_stop))
    lemas  = [token.lemma_ for token in doc_lema]

    print(f"Frase original: {frase}")
    print(f"Após remoção de stopwords: {sem_stop}")
    print(f"Stems (RSLP):  {stems}")
    print(f"Lemas (spaCy): {lemas}")
    print()

=== Comparação Stemming vs. Lematização ===

Frase original: Redes neurais profundas extraem representações hierárquicas de dados brutos
Após remoção de stopwords: ['redes', 'neurais', 'profundas', 'extraem', 'representações', 'hierárquicas', 'dados', 'brutos']
Stems (RSLP):  ['red', 'neur', 'profund', 'extra', 'represent', 'hierárqu', 'dad', 'brut']
Lemas (spaCy): ['rede', 'neural', 'profundo', 'extraem', 'representação', 'hierárquico', 'dado', 'bruto']

Frase original: A entrega demorou mais de uma hora e o pedido chegou completamente frio e sem condimentos
Após remoção de stopwords: ['entrega', 'demorou', 'hora', 'pedido', 'chegou', 'completamente', 'frio', 'condimentos']
Stems (RSLP):  ['entreg', 'demor', 'hor', 'ped', 'cheg', 'complet', 'fri', 'cond']
Lemas (spaCy): ['entregar', 'demorar', 'hora', 'pedir', 'chegar', 'completamente', 'frio', 'condimento']

Frase original: Não foi possível realizar o pagamento por aproximação devido à falha no terminal de vendas
Após remoção de stop

## 14. Bag of Words em Português com spaCy e NLTK

O scikit-learn aceita um tokenizador personalizado via o parâmetro `tokenizer` do `CountVectorizer`. Isso permite integrar o pipeline de pré-processamento baseado em spaCy diretamente à construção da MDT, combinando o melhor das duas bibliotecas — tal como recomendado pelo Prof. Bryan Kano nos slides da aula sobre NLTK e spaCy: "use o spaCy para transformar o texto em estrutura linguística e o NLTK para analisar estatisticamente essa estrutura".

In [32]:
# CountVectorizer com lematização via spaCy

cv_pt = CountVectorizer(
    tokenizer=lambda t: preprocessar_pt(t, usar_lema=True),
    token_pattern=None
)

# Treinamento e transformação
X_pt_count = cv_pt.fit_transform(sentences_pt)

# Vocabulário
vocab_pt = sorted(cv_pt.vocabulary_.keys())

# Informações gerais
print("Bag of Words em Português")
print(f"Quantidade de documentos: {X_pt_count.shape[0]}")
print(f"Tamanho do vocabulário: {X_pt_count.shape[1]}")
print(f"Formato da MDT: {X_pt_count.shape}")

# Primeiros tokens
print("\nPrimeiros 25 tokens do vocabulário:")
print(vocab_pt[:25])

# Frequência total dos tokens
frequencia_tokens = np.asarray(X_pt_count.sum(axis=0)).ravel()

df_vocab = pd.DataFrame({
    "Token": cv_pt.get_feature_names_out(),
    "Frequência": frequencia_tokens
})

df_vocab = df_vocab.sort_values(
    by="Frequência",
    ascending=False
)

print("\n10 tokens mais frequentes:")
print(df_vocab.head(10).to_string(index=False))

Bag of Words em Português
Quantidade de documentos: 25
Tamanho do vocabulário: 192
Formato da MDT: (25, 192)

Primeiros 25 tokens do vocabulário:
['Carbono', 'Vendas', 'abruptar', 'alto', 'analisar', 'antecedência', 'análise', 'apesar', 'aplicativo', 'aproximação', 'artesanal', 'atendimento', 'atleta', 'atualiza', 'avaliar', 'bag', 'belga', 'bilhão', 'borr', 'bruto', 'buraco', 'campo', 'captur', 'cardápio', 'cardíaco']

10 tokens mais frequentes:
      Token  Frequência
    cliente           5
       dado           2
   utilizar           2
   muscular           2
    maioria           2
     modelo           2
  gradiente           2
expectativa           2
 aplicativo           1
aproximação           1


In [33]:
# TF-IDF em português com o mesmo tokenizador
tfidf_pt = TfidfVectorizer(tokenizer=lambda t: preprocessar_pt(t, usar_lema=True),
                           token_pattern=None)
tfidf_pt.fit(sentences_pt)
X_pt_tfidf = tfidf_pt.transform(sentences_pt)

feature_pt = tfidf_pt.get_feature_names_out()
idf_pt     = tfidf_pt.idf_
idf_df_pt  = pd.Series(idf_pt, index=feature_pt).sort_values(ascending=False)

print("=== 10 tokens mais discriminativos em PT (maior IDF) ===")
print(idf_df_pt.head(10).to_string())
print()
print("=== 10 tokens menos discriminativos em PT (menor IDF) ===")
print(idf_df_pt.tail(10).to_string())

=== 10 tokens mais discriminativos em PT (maior IDF) ===
Carbono         3.564949
Vendas          3.564949
abruptar        3.564949
alto            3.564949
analisar        3.564949
antecedência    3.564949
análise         3.564949
apesar          3.564949
aplicativo      3.564949
aproximação     3.564949

=== 10 tokens menos discriminativos em PT (menor IDF) ===
trabalhar      3.564949
treino         3.564949
água           3.564949
dado           3.159484
modelo         3.159484
expectativa    3.159484
maioria        3.159484
muscular       3.159484
utilizar       3.159484
cliente        2.466337


## 15. Similaridade por Cosseno entre Frases em Português

A mesma análise de similaridade aplicada ao corpus em inglês é repetida aqui para o corpus em português, permitindo comparar como o pré-processamento com lematização afeta a proximidade semântica entre documentos.


In [34]:
sim_pt = cosine_similarity(X_pt_tfidf)

# Par mais similar
np.fill_diagonal(sim_pt, 0)
idx_max_pt = np.unravel_index(np.argmax(sim_pt), sim_pt.shape)
i_pt, j_pt = idx_max_pt
print(f"Par mais similar em PT (cos = {sim_pt[i_pt, j_pt]:.4f}):")
print(f"  Frase {i_pt+1}: {sentences_pt[i_pt]}")
print(f"  Frase {j_pt+1}: {sentences_pt[j_pt]}")
print()

# Par mais dissimilar
np.fill_diagonal(sim_pt, 1)
idx_min_pt = np.unravel_index(np.argmin(sim_pt), sim_pt.shape)
i2_pt, j2_pt = idx_min_pt
print(f"Par mais dissimilar em PT (cos = {sim_pt[i2_pt, j2_pt]:.4f}):")
print(f"  Frase {i2_pt+1}: {sentences_pt[i2_pt]}")
print(f"  Frase {j2_pt+1}: {sentences_pt[j2_pt]}")


Par mais similar em PT (cos = 0.1119):
  Frase 10: A nova sobremesa de chocolate belga superou as expectativas dos clientes mais exigentes
  Frase 14: Apesar do preço elevado o produto não correspondeu às expectativas dos consumidores

Par mais dissimilar em PT (cos = 0.0000):
  Frase 1: Redes neurais profundas extraem representações hierárquicas de dados brutos
  Frase 2: O gradiente descendente atualiza os pesos do modelo na direção oposta ao gradiente da perda


## 16. Análise Comparativa de Esparsidade

A esparsidade é uma propriedade central do BoW: quanto maior o vocabulário e mais distintos os documentos, mais zeros preencherão a matriz. Esta célula compara a esparsidade entre os diferentes vetorizadores e os dois idiomas, evidenciando como o pré-processamento (lematização, remoção de stopwords) reduz o vocabulário e, consequentemente, a esparsidade.


In [35]:
def calc_esparsidade(matriz):
    total = matriz.shape[0] * matriz.shape[1]
    return 1 - matriz.nnz / total

resultados_sparse = {
    "CountVectorizer (EN)":          calc_esparsidade(X_count),
    "TfidfVectorizer (EN)":          calc_esparsidade(X_tfidf),
    "HashingVectorizer (EN, 1024)":  calc_esparsidade(X_hash),
    "CountVectorizer+spaCy (PT)":    calc_esparsidade(X_pt_count),
    "TfidfVectorizer+spaCy (PT)":    calc_esparsidade(X_pt_tfidf),
}

print("=== Análise de Esparsidade por Vetorizador ===\n")
print(f"{'Vetorizador':<40} {'Esparsidade':>12} {'Shape':>15} {'Vocab':>8}")
print("-" * 80)

for nome, esp in resultados_sparse.items():
    if "EN" in nome and "Hash" not in nome:
        shape = X_count.shape if "Count" in nome else X_tfidf.shape
        vocab = len(cv.vocabulary_)
    elif "Hash" in nome:
        shape = X_hash.shape
        vocab = 1024
    else:
        shape = X_pt_count.shape if "Count" in nome else X_pt_tfidf.shape
        vocab = len(cv_pt.vocabulary_)
    print(f"{nome:<40} {esp:>11.2%} {str(shape):>15} {vocab:>8}")


=== Análise de Esparsidade por Vetorizador ===

Vetorizador                               Esparsidade           Shape    Vocab
--------------------------------------------------------------------------------
CountVectorizer (EN)                          95.16%       (25, 239)       17
TfidfVectorizer (EN)                          95.16%       (25, 239)       17
HashingVectorizer (EN, 1024)                  98.88%      (25, 1024)     1024
CountVectorizer+spaCy (PT)                    95.79%       (25, 192)      192
TfidfVectorizer+spaCy (PT)                    95.79%       (25, 192)      192


## 17. Análise de Frequência e Lei de Zipf

A Lei de Zipf estabelece que a frequência de uma palavra é inversamente proporcional ao seu rank no vocabulário ordenado por frequência. Isso significa que poucas palavras são muito frequentes (e geralmente pouco informativas) enquanto a grande maioria das palavras é rara. Esse fenômeno justifica tanto a remoção de stopwords quanto o uso de TF-IDF, que penaliza palavras ubíquas no corpus.


In [40]:
# Recriando o CountVectorizer original em inglês

cv_en = CountVectorizer()

X_count = cv_en.fit_transform(sentences_en)

# Frequência total dos tokens em inglês

freq_total_en = np.asarray(
    X_count.sum(axis=0)
).ravel()

feature_names_en = cv_en.get_feature_names_out()

freq_series_en = pd.Series(
    freq_total_en,
    index=feature_names_en
).sort_values(ascending=False)

print("Top 20 tokens mais frequentes em inglês:")
print(freq_series_en.head(20).to_string())


# Frequência total dos tokens em português

freq_total_pt = np.asarray(
    X_pt_count.sum(axis=0)
).ravel()

feature_names_pt = cv_pt.get_feature_names_out()

freq_series_pt = pd.Series(
    freq_total_pt,
    index=feature_names_pt
).sort_values(ascending=False)

print("\nTop 20 tokens mais frequentes em português:")
print(freq_series_pt.head(20).to_string())

Top 20 tokens mais frequentes em inglês:
the         21
and          8
to           7
of           5
in           3
as           3
by           3
is           3
not          3
that         3
precise      2
allows       2
be           2
cold         2
data         2
model        2
water        2
requires     2
often        2
wall         2

Top 20 tokens mais frequentes em português:
cliente         5
dado            2
utilizar        2
muscular        2
maioria         2
modelo          2
gradiente       2
expectativa     2
aplicativo      1
aproximação     1
abruptar        1
Carbono         1
analisar        1
antecedência    1
análise         1
Vendas          1
bag             1
belga           1
borr            1
bilhão          1


## 18. Quadro Comparativo dos Vetorizadores

A tabela abaixo sintetiza as características dos métodos aplicados neste notebook, facilitando a escolha do mais adequado para cada contexto de aplicação.

| Vetorizador | Escala | Vocabulário | Interpretável | Indicado para |
|---|---|---|---|---|
| `CountVectorizer` | Pequeno/Médio | Explícito | Sim | Exploração, Naïve Bayes |
| `TfidfVectorizer` | Pequeno/Médio | Explícito | Sim | Classificação, busca |
| `HashingVectorizer` | Grande | Implícito | Não | Streaming, grande escala |
| Keras `Tokenizer` | Pequeno/Médio | Explícito | Sim | Redes neurais, LSTM |
| NLTK + RSLP (PT) | Qualquer | Explícito | Sim | Português, prototipagem |
| spaCy `pt_core_news_sm` | Qualquer | Explícito | Sim | Português, produção |

A combinação spaCy (lematização) + scikit-learn (vetorização) representa a abordagem de maior qualidade para o processamento em português, pois preserva formas canônicas válidas no léxico e permite a aplicação direta de qualquer modelo de ML do scikit-learn sobre os vetores resultantes.


## 19. Conclusão

Este notebook implementou, de forma progressiva e comentada, o modelo Bag of Words em suas principais variantes disponíveis nos ecossistemas scikit-learn e Keras, aplicado a dois conjuntos de 25 frases nos idiomas inglês e português.

Os principais aprendizados consolidados ao longo das seções foram os seguintes.

O `CountVectorizer` é o ponto de partida mais simples e interpretável: cada posição do vetor corresponde a uma palavra do vocabulário e o valor é a contagem de ocorrências. A Matriz Documento-Termo resultante é a entrada direta para algoritmos de classificação supervisionada como Naïve Bayes e SVM.

O `TfidfVectorizer` aprimora o CountVectorizer ao ponderar a frequência local pelo logaritmo inverso da frequência documental, elevando palavras discriminativas e rebaixando palavras ubíquas. Isso o torna mais robusto para tarefas de busca e classificação, especialmente quando o corpus é heterogêneo em domínio.

O `HashingVectorizer` sacrifica a interpretabilidade em troca de escalabilidade: ao dispensar o vocabulário explícito, pode processar corpora de qualquer tamanho em fluxo contínuo, sem armazenar o vocabulário na memória.

O Keras `Tokenizer` integra-se naturalmente a arquiteturas de redes neurais, oferecendo quatro modos de codificação (binário, contagem, tfidf, frequência) e sendo o passo anterior ao embedding lookup.

Para o português, a combinação NLTK (stopwords) + spaCy (lematização neural) demonstrou qualidade superior ao stemming com RSLP, produzindo formas canônicas válidas no léxico e reduzindo o vocabulário de forma mais precisa. A análise comparativa de esparsidade confirmou que o pré-processamento adequado reduz substancialmente o tamanho do vocabulário, tornando os vetores menos esparsos e os modelos mais eficientes.

**Referências Bibliográficas**

Bird, S., Klein, E. & Loper, E. (2009). *Natural Language Processing with Python*. O'Reilly Media.

Manning, C., Raghavan, P. & Schütze, H. (2008). *Introduction to Information Retrieval*. Cambridge University Press.

Gandhi, V. (2020). Bag-of-Words Model for Beginners. *Kaggle*. Disponível em: kaggle.com/code/vipulgandhi/bag-of-words-model-for-beginners
